In [1]:
import random
import copy
from collections import deque
from IPython.display import clear_output
import time
import pygame as pg

pygame 2.6.1 (SDL 2.28.4, Python 3.13.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


c:\Users\adilo\OneDrive\Desktop\Code\code\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


# Main Implementation

In [2]:
class Vehicule:
    def __init__(self, position, orientation,length):
        self.position = position
        self.orientation = orientation
        self.length = length
        
    
    def displayVehiculeInfo(self):
        print(f"Vehicule stats: \nPosition: {self.position}\nOrientation: {self.orientation}\nLength {self.length}")
    
    def getVehiculeSpace(self):
        occupiedPixels = []
        if self.orientation == 'horizontal':
            for i in range(0, self.length):
                occupiedPixels.append((self.position[0], self.position[1]+i))
        else:
            for i in range(0, self.length):
                occupiedPixels.append((self.position[0]+i, self.position[1]))   
        return occupiedPixels
    
    def moveVehiculeByCoords(self, move):
        self.position[0] += move[0]
        self.position[1] += move[1]
        
    def moveVehicule(self, direction, step):
        if self.orientation == 'vertical':
            if direction.lower() == 'up':
                x = self.position[0] - step
                if x >= 0:
                    newPosition = (x, self.position[1])
                    self.position = newPosition
                else:
                    print('Position out of bound')
                
            elif direction.lower() == 'down':
                x = self.position[0] + step
                if x + self.length - 1 <= 5:
                    newPosition = (x, self.position[1])
                    self.position = newPosition
                else:
                    print('Position out of bound')

        elif self.orientation == 'horizontal':

            if direction.lower() == 'left':
                    y = self.position[1] - step
                    if y >= 0:
                        newPosition = (self.position[0], y)
                        self.position = newPosition
                    else:
                        print('Position out of bound') 
                    
            elif direction.lower() == 'right':
                y = self.position[1] + step
                if y + self.length - 1 <= 5:
                    newPosition = (self.position[0], y)
                    self.position = newPosition
                else:
                    print('Position out of bound')

        else:
            print('Vehicule has broken Orientation !')


In [3]:
def defineNumberOfObstacles(difficulty = 'hard'):
    if difficulty == 'easy':
        nbrObs = random.randint(3, 5)
    elif difficulty == 'medium':
        nbrObs = random.randint(5, 8)
    elif difficulty == 'hard':
        nbrObs = random.randint(8, 9)
    else:
        print("Unknown difficulty !")
        nbrObs = None
    return nbrObs

def generateVehicule():
    length = random.randint(2,3)
    bin = random.randint(0,1)
    if bin == 0:
        orientation = 'horizontal'
    else:
        orientation = 'vertical'
    
    if bin == 0:
        x = random.randint(0,5)
        while x == 2:
            x = random.randint(0,5)
        position = ( x, random.randint(0, 5-length)) 
    else:
        position = (random.randint(0,5-length), random.randint(0,5))
    
    return Vehicule(orientation = orientation, position = position, length =length)    

In [4]:
def gridToMatrixDrawer(grid):
    #       0 1 2 3 4 5
    mat = [[0,0,0,0,0,0], #0
           [0,0,0,0,0,0], #1
           [0,0,0,0,0,0], #2
           [0,0,0,0,0,0], #3
           [0,0,0,0,0,0], #4
           [0,0,0,0,0,0]] #5
    
    for pixel in grid.redCar.getVehiculeSpace():
        mat[pixel[0]][pixel[1]] = 'x'
    
    for i, vhc in enumerate(grid.obstacles):
        for pixel in vhc.getVehiculeSpace():
            if mat[pixel[0]][pixel[1]] != 0:
                print(f"Conflict spotted at: ({pixel[0]},{pixel[1]})")
            mat[pixel[0]][pixel[1]] = i+1
    return mat
    

In [5]:
class Grid:
    def __init__(self, redCar, obstacles):
        self.redCar = redCar
        self.obstacles = obstacles
        self.matrix = gridToMatrixDrawer(self)
        self.parent = None
        self.objectiveValue = None
    
    def setParent(self, parent):
        self.parent = parent

    def gridToMatrixDrawer(self):
        #       0 1 2 3 4 5
        mat = [[0,0,0,0,0,0], #0
               [0,0,0,0,0,0], #1
               [0,0,0,0,0,0], #2
               [0,0,0,0,0,0], #3
               [0,0,0,0,0,0], #4
               [0,0,0,0,0,0]] #5
        
        for pixel in self.redCar.getVehiculeSpace():
            mat[pixel[0]][pixel[1]] = 'x'
        
        for i, vhc in enumerate(self.obstacles):
            for pixel in vhc.getVehiculeSpace():
                if mat[pixel[0]][pixel[1]] != 0:
                    print(f"Conflict spotted at: ({pixel[0]},{pixel[1]})")
                mat[pixel[0]][pixel[1]] = i+1
        self.matrix = mat

    def displayGridInfo(self):
        print(f"Grid Stats: \n")
        print('================================')
        self.redCar.displayVehiculeInfo()
        print(f'Red Car: \n')
        print('================================')
        print(f'Obstacles: {len(self.obstacles)}\n')
        for vhc in self.obstacles:
            vhc.displayVehiculeInfo()
        print('================================')
        print(f'Matrix: \n')
        print(self.matrix)
    
    def showMatrix(self):
        print(self.matrix)

    def spaceAroundVehicule(self, vhc):
        occupiedPixels = vhc.getVehiculeSpace()
        start = occupiedPixels[0]
        end = occupiedPixels[len(occupiedPixels)-1]
        if vhc.orientation == 'horizontal':
            i = start[0]
            j = start[1] - 1
            left = 0
            right = 0
            while j >= 0 and self.matrix[i][j] == 0 :
                left +=1
                j -= 1
            i = end[0]
            j = end[1] + 1
            while j <= 5 and self.matrix[i][j] == 0:
                right +=1
                j += 1
            return vhc.orientation ,left, right
        elif vhc.orientation == 'vertical':
            i = start[0] - 1
            j = start[1] 
            up = 0
            down = 0
            while i >= 0 and self.matrix[i][j] == 0 :
                up +=1
                i -= 1
            i = end[0] + 1
            j = end[1] 
            while i <= 5 and self.matrix[i][j] == 0 :
                down +=1
                i += 1
            return vhc.orientation ,up, down
                
        else:
            print('Vehicule has Broken orientation')
            return None
    
    def makeChildsFromVehicule(self, vhc, idx = None):
        children = []
        orientation, before, after = self.spaceAroundVehicule(vhc)
        if orientation == 'horizontal':
            for i in range(before):
                vhc2 = Vehicule(vhc.position, vhc.orientation, vhc.length)
                vhc2.moveVehicule('left', i+1)
                if idx == None:
                    children.append(Grid(vhc2, self.obstacles))
                else:
                    obstacles2 = copy.deepcopy(self.obstacles)
                    obstacles2[idx] = vhc2
                    children.append(Grid(self.redCar,obstacles2))
                vhc2 = None

            for i in range(after):
                vhc2 = Vehicule(vhc.position, vhc.orientation, vhc.length)
                vhc2.moveVehicule('right', i+1)
                if idx == None:
                    children.append(Grid(vhc2, self.obstacles))
                else:
                    obstacles2 = copy.deepcopy(self.obstacles)
                    obstacles2[idx] = vhc2
                    children.append(Grid(self.redCar,obstacles2))
                vhc2 = None
        elif orientation == 'vertical':
            for i in range(before):
                vhc2 = Vehicule(vhc.position, vhc.orientation, vhc.length)
                vhc2.moveVehicule('up', i+1)
                if idx == None:
                    children.append(Grid(vhc2, self.obstacles))
                else:
                    obstacles2 = copy.deepcopy(self.obstacles)
                    obstacles2[idx] = vhc2
                    children.append(Grid(self.redCar,obstacles2))
                vhc2 = None
            for i in range(after):
                vhc2 = Vehicule(vhc.position, vhc.orientation, vhc.length)
                vhc2.moveVehicule('down', i+1)
                if idx == None:
                    children.append(Grid(vhc2, self.obstacles))
                else:
                    obstacles2 = copy.deepcopy(self.obstacles)
                    obstacles2[idx] = vhc2
                    children.append(Grid(self.redCar,obstacles2))
                vhc2 = None
        return children   

    def allPossibleMoves(self):
        redCarMoves = self.makeChildsFromVehicule(self.redCar)
        obstaclesMoves = []
        for i, vhc in enumerate(self.obstacles):
            obstaclesMoves.extend(self.makeChildsFromVehicule(vhc, idx = i))
        return redCarMoves + obstaclesMoves




# Main Functions Phase

In [6]:
#       0 1 2 3 4 5
mat = [[0,1,0,0,0,0], #0
       [0,0,0,0,0,0], #1
       [0,0,0,0,0,0], #2
       [0,0,0,0,0,0], #3
       [0,0,0,0,0,0], #4
       [0,0,0,0,0,0]] #5

In [7]:
def gridRandomGenerator(difficulty = 'hard', nbrObs = None):
    if not nbrObs:
        nbrObs = defineNumberOfObstacles(difficulty)
    filledSpace = []
    obstacles = []

    redCar = Vehicule((2,0), 'horizontal', 2)
    x = [pixel for pixel in redCar.getVehiculeSpace()]
    for pixel in x:
        filledSpace.append(pixel)

    print(f'will generate {nbrObs} obstacle')
    for _ in range(nbrObs):
        vhc = generateVehicule()
        y = [pixel for pixel in vhc.getVehiculeSpace()]
        if any(pixel in filledSpace for pixel in y):
            del vhc
        else:
            obstacles.append(vhc)
            for pixel in y:
                filledSpace.append(pixel)
    print(f'generated {nbrObs} obstacle')
    return Grid(redCar, obstacles)


In [8]:
def compareVehicules(vhc, vhc2):
    if vhc.position == vhc2.position and vhc.orientation == vhc2.orientation and vhc.length == vhc2.length:
        return True
    else:
        return False

In [9]:
def compareGrids(grid, grid2):
    indicator = False
    if compareVehicules(grid.redCar, grid2.redCar) and len(grid.obstacles) == len(grid2.obstacles):
        for i in range(len(grid.obstacles)):
            if not compareVehicules(grid.obstacles[i], grid2.obstacles[i]):
                return False
        return True
    else:
        return False

In [10]:
def movesInAllMoves(list1, list2):
    indicator = False
    for i, grid in enumerate(list1):
        for grid2 in list2:
            if compareGrids(grid, grid2):
                indicator = True
                break
        if indicator:
            indicator = False
        else:
            return False
            
    return True

In [11]:
def finalState(grid):
    if grid.redCar.position[1]+ 1 + grid.spaceAroundVehicule(grid.redCar)[2] == 5:
        return True
    else:
        return False

# Unit Tests Area

In [16]:
els = gridRandomGenerator()
print(els.parent)
els.setParent("It worked !")
print(els.parent)

None
It worked !


In [17]:
sv = gridRandomGenerator()
sm = gridRandomGenerator()
s1 = gridRandomGenerator()
s2 = gridRandomGenerator()
s3 = gridRandomGenerator()
s3 = gridRandomGenerator()

In [18]:
s4 = gridRandomGenerator()

In [19]:
compareGrids(sm,s3)

False

In [20]:
movesInAllMoves([sm,sv,s4],[s4, s1,s2,sm,s3,sv])

True

In [21]:
gridToMatrixDrawer(grid)

NameError: name 'grid' is not defined

In [ ]:
s.redCar.moveVehicule('right',1)

In [ ]:
compareGrids(grid, s)

True

In [ ]:
movesInAllMoves([s,sv],[els, grid])

stopping at 1


False

In [ ]:
g.obstacles[1].moveVehicule('left',1)

In [ ]:
gridToMatrixDrawer(ge)

[[0, 5, 0, 1, 1, 0],
 [0, 5, 0, 0, 0, 0],
 ['x', 'x', 0, 0, 6, 0],
 [0, 0, 0, 0, 6, 0],
 [0, 3, 3, 4, 4, 0],
 [0, 0, 2, 2, 0, 0]]

In [ ]:
compareGrids(g,grid)

False

In [ ]:
movesInAllMoves([g],[grid, ge])

True

In [ ]:
movesInAllMoves([g],[grid, ge])

True

In [ ]:
finalState(g)

False

In [ ]:
grid = gridRandomGenerator()
len(grid.obstacles)

6

In [ ]:
gridToMatrixDrawer(grid)

[[2, 2, 2, 0, 0, 0],
 [0, 0, 0, 0, 0, 0],
 ['x', 'x', 0, 0, 0, 0],
 [0, 1, 1, 0, 0, 0],
 [0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0]]

In [ ]:
allMoves = grid.allPossibleMoves()

In [ ]:
if movesInAllMoves(grid.makeChildsFromVehicule(grid.redCar), allMoves):
     print(f'redCar All good so far')
else:
     print(f'Stopped at redCar')
     
for i in range(len(grid.obstacles)):
     if movesInAllMoves(grid.makeChildsFromVehicule(grid.redCar), allMoves):
          print(f'obstacle {i} All good so far')
     else:
          print(f'Stopped at obstacle {i}')
          break


redCar All good so far
obstacle 0 All good so far
obstacle 1 All good so far


# Convertion to PyGame objects

In [12]:
grid = gridRandomGenerator()

will generate 9 obstacle
generated 9 obstacle


In [13]:
grid.showMatrix()

[[0, 0, 0, 1, 1, 0], [0, 0, 0, 0, 0, 0], ['x', 'x', 0, 0, 2, 0], [0, 0, 4, 0, 2, 0], [0, 0, 4, 0, 0, 0], [0, 0, 3, 3, 3, 0]]


In [14]:
grid.displayGridInfo()

Grid Stats: 

Vehicule stats: 
Position: (2, 0)
Orientation: horizontal
Length 2
Red Car: 

Obstacles: 4

Vehicule stats: 
Position: (0, 3)
Orientation: horizontal
Length 2
Vehicule stats: 
Position: (2, 4)
Orientation: vertical
Length 2
Vehicule stats: 
Position: (5, 2)
Orientation: horizontal
Length 3
Vehicule stats: 
Position: (3, 2)
Orientation: vertical
Length 2
Matrix: 

[[0, 0, 0, 1, 1, 0], [0, 0, 0, 0, 0, 0], ['x', 'x', 0, 0, 2, 0], [0, 0, 4, 0, 2, 0], [0, 0, 4, 0, 0, 0], [0, 0, 3, 3, 3, 0]]


In [46]:
def gridToRects(grid, board = None, margin = 5):
    if board:
        x = board.x
        y = board.y
    else:
        x = 100
        y = 100

    obstacles = []
    for i, obstacle in enumerate(grid.obstacles):
        if obstacle.orientation == 'horizontal':
            print(f' obstacle {i} ==> i: {x} + {obstacle.position[1]}*100 + {obstacle.position[1]+1}*{margin}  / j: {y} + {obstacle.position[0]}*100 + {obstacle.position[0]+1}*{margin} ')
            obstacles.append(pg.Rect(x + obstacle.position[1]*100 + (obstacle.position[1] + 1)*margin, y + obstacle.position[0]*100 + (obstacle.position[0] + 1)*margin, obstacle.length*100,100))
        elif obstacle.orientation == 'vertical':
            print(f'obstacle {i} ==> skip')
            # obstacles.append(pg.Rect(x + obstacle.position[0]*100 + (obstacle.position[0] + 1)*margin, y + obstacle.position[1]*100 + (obstacle.position[1] + 1)*margin, 100, obstacle.length*100))
        else:
            print(f'Vehicule {i} has broken orientation')
    return(obstacles)

In [27]:
def gridsToMove(grid1, grid2):
    if not compareVehicules(grid.redCar, grid2.redCar):
        return ('redCar',100 * (grid2.redCar.position[0] - grid1.redCar.position[0]), 100 * (grid2.redCar.position[1] - grid1.redCar.position[1]))
    else:
        for i in range(len(grid1.obstacles)):
            if not compareVehicules(grid1.obstacles[i], grid2.obstacles[i]):
                return (i, grid2.obstacles[i].position[0] - grid1.obstacles[i].position[0], grid2.obstacles[i].position[1] - grid1.obstacles[i].position[1])
        print(f'No move to make !')
        return None

In [ ]:
def moveVehiculeByCoords(vehicule, move):
    vehicule.position[0] += move[0]
    vehicule.position[1] += move[1]

In [ ]:
def displayGame(solution, disRedCar, disSolution):
    actual = solution[0]
    for i in range(1, len(solution)):
        move = gridsToMove(actual, grid)
        if move[0] == 'redCar':
            actual.redCar.moveVehiculeByCoords((move[1],move[2]))
            disRedCar.x += move[1]*100
            disRedCar.y += move[2]*100
        else:
            actual.obstacles[i].moveVehiculeByCoords((move[1],move[2]))
            disSolution[i].x += move[1]*100
            disSolution[i].y += move[2]*100       

## Unit Tests

In [48]:
# Board rectangle
margin = 5
board = pg.Rect(100,100, 600 + 7*margin, 600 + 7*margin)

In [49]:
grid = gridRandomGenerator()
displayMatrixGrid(grid.matrix)

will generate 8 obstacle
generated 8 obstacle
4 4 4 3 0 0
0 0 0 3 0 0
x x 0 3 0 0
0 0 0 0 0 0
0 1 1 1 0 0
2 2 0 0 0 0


In [50]:
gridToRects(grid, board, margin)

 obstacle 0 ==> i: 100 + 1*100 + 2*5  / j: 100 + 4*100 + 5*5 
 obstacle 1 ==> i: 100 + 0*100 + 1*5  / j: 100 + 5*100 + 6*5 
obstacle 2 ==> skip
 obstacle 3 ==> i: 100 + 0*100 + 1*5  / j: 100 + 0*100 + 1*5 


[<rect(210, 525, 300, 100)>,
 <rect(105, 630, 200, 100)>,
 <rect(105, 105, 300, 100)>]

## Unit Tests

## Unit Tests

# AI Solvers

In [22]:
def BFSsolver(rootGrid):
    queue = deque()
    visited = []
    queue.append(rootGrid)
    cpt = 0
    while queue:
        head = queue.popleft()
        print(f' visited so far:           {len(visited)}')
        if finalState(head):
            print('End of search, Puzzle solved !')
            return(head)
        elif not movesInAllMoves([head], visited):
            visited.append(head)
            sons = head.allPossibleMoves()
            for son in sons:
                son.setParent(head)
                queue.append(son)
            print(f'Sons of head {cpt} : {len(sons)}')
        else:
            print('No win, already visited !')
    print('Broke out of the loop, no solution')
    return None



In [23]:
def DFSsolver(rootGrid):
    queue = deque()
    visited = []
    queue.append(rootGrid)
    cpt = 0
    while queue:
        head = queue.pop()
        print(f' visited so far:           {len(visited)}')
        if finalState(head):
            print('End of search, Puzzle solved !')
            return(head)
        elif not movesInAllMoves([head], visited):
            visited.append(head)
            sons = head.allPossibleMoves()
            for son in sons:
                son.setParent(head)
                queue.append(son)
            print(f'Sons of head {cpt} : {len(sons)}')
        else:
            print('No win, already visited !')
    print('Broke out of the loop, no solution')
    return None



# Unit tests Area

In [250]:
rootGrid = gridRandomGenerator()
gridToMatrixDrawer(rootGrid)

will generate 8 obstacle
generated 8 obstacle


[[0, 2, 2, 2, 0, 0],
 [0, 0, 0, 0, 0, 4],
 ['x', 'x', 3, 0, 0, 4],
 [0, 0, 3, 0, 0, 0],
 [1, 1, 5, 5, 5, 0],
 [0, 0, 0, 0, 0, 0]]

In [251]:
BFSsolution = BFSsolver(rootGrid)

 visited so far:           0
Sons of head 0 : 9
 visited so far:           1
Sons of head 0 : 9
 visited so far:           2
Sons of head 0 : 9
 visited so far:           3
Sons of head 0 : 9
 visited so far:           4
Sons of head 0 : 9
 visited so far:           5
Sons of head 0 : 8
 visited so far:           6
Sons of head 0 : 9
 visited so far:           7
Sons of head 0 : 8
 visited so far:           8
Sons of head 0 : 8
 visited so far:           9
Sons of head 0 : 10
 visited so far:           10
No win, already visited !
 visited so far:           10
No win, already visited !
 visited so far:           10
No win, already visited !
 visited so far:           10
Sons of head 0 : 9
 visited so far:           11
Sons of head 0 : 8
 visited so far:           12
Sons of head 0 : 9
 visited so far:           13
Sons of head 0 : 8
 visited so far:           14
Sons of head 0 : 8
 visited so far:           15
Sons of head 0 : 10
 visited so far:           16
No win, already visited !


In [252]:
gridToMatrixDrawer(rootGrid)

[[0, 2, 2, 2, 0, 0],
 [0, 0, 0, 0, 0, 4],
 ['x', 'x', 3, 0, 0, 4],
 [0, 0, 3, 0, 0, 0],
 [1, 1, 5, 5, 5, 0],
 [0, 0, 0, 0, 0, 0]]

In [253]:
gridToMatrixDrawer(BFSsolution)

[[0, 0, 3, 2, 2, 2],
 [0, 0, 3, 0, 0, 0],
 ['x', 'x', 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 4],
 [1, 1, 5, 5, 5, 4],
 [0, 0, 0, 0, 0, 0]]

In [254]:
DFSsolution = DFSsolver(rootGrid)

 visited so far:           0
Sons of head 0 : 9
 visited so far:           1
Sons of head 0 : 10
 visited so far:           2
No win, already visited !
 visited so far:           2
Sons of head 0 : 10
 visited so far:           3
Sons of head 0 : 9
 visited so far:           4
No win, already visited !
 visited so far:           4
Sons of head 0 : 8
 visited so far:           5
Sons of head 0 : 8
 visited so far:           6
Sons of head 0 : 9
 visited so far:           7
No win, already visited !
 visited so far:           7
No win, already visited !
 visited so far:           7
No win, already visited !
 visited so far:           7
End of search, Puzzle solved !


In [ ]:
gridToMatrixDrawer(rootGrid)

[[0, 2, 2, 2, 0, 0],
 [0, 0, 0, 0, 0, 4],
 ['x', 'x', 3, 0, 0, 4],
 [0, 0, 3, 0, 0, 0],
 [1, 1, 5, 5, 5, 0],
 [0, 0, 0, 0, 0, 0]]

In [256]:
gridToMatrixDrawer(DFSsolution)

[[0, 2, 2, 2, 0, 4],
 [0, 0, 0, 0, 0, 4],
 ['x', 'x', 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0],
 [1, 1, 3, 5, 5, 5],
 [0, 0, 3, 0, 0, 0]]

# Infomred Research Algorithms

In [170]:
def weight(root, grid):
    if compareGrids(root,grid):
        return 0
    else:
        cpt = 0
        for i in range(len(grid.obstacles)):
            if not compareVehicules(root.obstacles[i], grid.obstacles[i]):
                cpt += 1
        if not compareVehicules(root.redCar, grid.redCar):
            cpt += 1
        return cpt

In [102]:
def heuristic(grid):
    values = []
    if grid.redCar.position[1] == 4:
        return 0
    for j in range(grid.redCar.position[1]+2 ,6):
        values.append(grid.matrix[2][j])
    vals = set(values)
    if 0 in vals:
        vals.remove(0)
    return len(vals)

In [171]:
def setObjectiveValue(root, grid):
    grid.objectiveValue = weight(root, grid) + heuristic(grid)
    return grid

In [183]:
def reorder(movesList):
    for i in range(len(movesList)):
        for j in range(i+1, len(movesList)):
            if movesList[i].objectiveValue < movesList[j].objectiveValue:
                x = movesList[j].objectiveValue
                movesList[j].objectiveValue = movesList[i].objectiveValue
                movesList[i].objectiveValue = x
    return movesList
    

In [263]:
def Astar(rootGrid):
    queue = []
    visited = []
    setObjectiveValue(rootGrid, rootGrid)
    queue.append(rootGrid)
    cpt = 0
    while queue:
        reorder(queue)
        head = queue[0]
        queue.remove(head)
        print(f' visited so far:           {len(visited)}')
        if finalState(head):
            print('End of search, Puzzle solved !')
            return(head)
        elif not movesInAllMoves([head], visited):
            visited.append(head)
            sons = head.allPossibleMoves()
            for son in sons:
                son.setParent(head)
                setObjectiveValue(rootGrid, son)
                queue.append(son)
            print(f'Sons of head {cpt} : {len(sons)}')
        else:
            print('No win, already visited !')
    print('Broke out of the loop, no solution')
    return None

In [16]:
def displayMatrixGrid(matrix):
    for row in matrix:
        print(' '.join(str(x) for x in row))

# Unit Tests Area

In [95]:
root = gridRandomGenerator()
gridToMatrixDrawer(root)


[[0, 1, 1, 0, 4, 0],
 [0, 0, 0, 3, 4, 5],
 ['x', 'x', 2, 3, 4, 5],
 [0, 0, 2, 3, 0, 0],
 [0, 0, 2, 0, 0, 0],
 [6, 6, 0, 0, 0, 0]]

In [96]:
grid = copy.deepcopy(root)
gridToMatrixDrawer(grid)

[[0, 1, 1, 0, 4, 0],
 [0, 0, 0, 3, 4, 5],
 ['x', 'x', 2, 3, 4, 5],
 [0, 0, 2, 3, 0, 0],
 [0, 0, 2, 0, 0, 0],
 [6, 6, 0, 0, 0, 0]]

In [97]:
grid.redCar.moveVehicule('left',2)
gridToMatrixDrawer(grid)

Position out of bound


[[0, 1, 1, 0, 4, 0],
 [0, 0, 0, 3, 4, 5],
 ['x', 'x', 2, 3, 4, 5],
 [0, 0, 2, 3, 0, 0],
 [0, 0, 2, 0, 0, 0],
 [6, 6, 0, 0, 0, 0]]

In [98]:
grid.obstacles[2].moveVehicule('down', 1)
grid.matrix = gridToMatrixDrawer(grid)
grid.matrix

[[0, 1, 1, 0, 4, 0],
 [0, 0, 0, 0, 4, 5],
 ['x', 'x', 2, 3, 4, 5],
 [0, 0, 2, 3, 0, 0],
 [0, 0, 2, 3, 0, 0],
 [6, 6, 0, 0, 0, 0]]

In [172]:
weight(root, grid)

1

In [173]:
heuristic(grid)

4

In [174]:
val.objectiveValue

5

In [175]:
val = setObjectiveValue(root, grid)
val.objectiveValue

5

In [240]:
gridRandomGenerator(nbrObs= 0).obstacles

will generate 9 obstacle
generated 9 obstacle


In [241]:
len(root.obstacles)

6

In [243]:
valsList = []
for i in range(6):
    temp = gridRandomGenerator(nbrObs = 6)
    print(f'root {len(root.obstacles)} / grid {len(temp.obstacles)}')
    temp = setObjectiveValue(root,temp)
    valsList.append(temp)

for i in range(6):
    print(valsList[i].objectiveValue)



will generate 6 obstacle
generated 6 obstacle
root 6 / grid 5
will generate 6 obstacle
generated 6 obstacle
root 6 / grid 4
will generate 6 obstacle
generated 6 obstacle
root 6 / grid 3
will generate 6 obstacle
generated 6 obstacle
root 6 / grid 3
will generate 6 obstacle
generated 6 obstacle
root 6 / grid 2
will generate 6 obstacle
generated 6 obstacle
root 6 / grid 5
5
4
5
5
3
8


In [244]:
res =reorder(valsList)
for i in range(len(res)):
    print(res[i].objectiveValue)

8
5
5
5
4
3


In [140]:
valsList = []
for i in range(5):
    temp = gridRandomGenerator()
    valsList.append(temp)
    print(f'root {len(root.obstacles)} / grid {len(grid.obstacles)}')

root 6 / grid 6
root 6 / grid 6
root 6 / grid 6
root 6 / grid 6
root 6 / grid 6


In [201]:
for i in range(5):
    print(len(valsList[i].obstacles))

4
4
4
3
3


In [257]:
valsList

In [264]:
Asolution = Astar(rootGrid)
gridToMatrixDrawer(rootGrid)

 visited so far:           0
Sons of head 0 : 9
 visited so far:           1
Sons of head 0 : 9
 visited so far:           2
Sons of head 0 : 9
 visited so far:           3
Sons of head 0 : 9
 visited so far:           4
Sons of head 0 : 9
 visited so far:           5
Sons of head 0 : 8
 visited so far:           6
Sons of head 0 : 9
 visited so far:           7
Sons of head 0 : 8
 visited so far:           8
Sons of head 0 : 8
 visited so far:           9
Sons of head 0 : 10
 visited so far:           10
No win, already visited !
 visited so far:           10
No win, already visited !
 visited so far:           10
No win, already visited !
 visited so far:           10
Sons of head 0 : 9
 visited so far:           11
Sons of head 0 : 8
 visited so far:           12
Sons of head 0 : 9
 visited so far:           13
Sons of head 0 : 8
 visited so far:           14
Sons of head 0 : 8
 visited so far:           15
Sons of head 0 : 10
 visited so far:           16
No win, already visited !


[[0, 2, 2, 2, 0, 0],
 [0, 0, 0, 0, 0, 4],
 ['x', 'x', 3, 0, 0, 4],
 [0, 0, 3, 0, 0, 0],
 [1, 1, 5, 5, 5, 0],
 [0, 0, 0, 0, 0, 0]]

In [265]:
gridToMatrixDrawer(BFSsolution)

[[0, 0, 3, 2, 2, 2],
 [0, 0, 3, 0, 0, 0],
 ['x', 'x', 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 4],
 [1, 1, 5, 5, 5, 4],
 [0, 0, 0, 0, 0, 0]]

In [266]:
gridToMatrixDrawer(DFSsolution)

[[0, 2, 2, 2, 0, 4],
 [0, 0, 0, 0, 0, 4],
 ['x', 'x', 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0],
 [1, 1, 3, 5, 5, 5],
 [0, 0, 3, 0, 0, 0]]

In [267]:
gridToMatrixDrawer(Asolution)

[[0, 0, 3, 2, 2, 2],
 [0, 0, 3, 0, 0, 0],
 ['x', 'x', 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 4],
 [1, 1, 5, 5, 5, 4],
 [0, 0, 0, 0, 0, 0]]

# Display

In [313]:
def solvePuzzle(solution):  
    win = solution  
    finalMoves = deque()
    while solution.parent != None:
        finalMoves.appendleft(solution)
        solution = solution.parent
    while finalMoves:
        move = finalMoves.popleft()
        displayMatrixGrid(move.matrix)
        time.sleep(1)
        clear_output(wait = True)
    displayMatrixGrid(win.matrix)

In [314]:
solvePuzzle(Asolution)

0 0 3 2 2 2
0 0 3 0 0 0
x x 0 0 0 0
0 0 0 0 0 4
1 1 5 5 5 4
0 0 0 0 0 0
